## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Optuna
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/"data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [7]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [8]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [9]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [10]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [11]:
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])

In [12]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [13]:
def objective_optuna(trial):
  weights = [trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(num_models)]
  sum_w = sum(weights)
  if sum_w == 0:
    norm_w = weights
  else:
    norm_w = [w / sum_w for w in weights]

  blend = np.dot(X.values, norm_w)
  return roc_auc_score(y, blend)  # Optuna maximizes by default

In [14]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_optuna, n_trials=1500)

[I 2026-11-10 16:38:23,758] A new study created in memory with name: no-name-c5b4d983-259b-42a1-a61b-c8b68103093f
[I 2026-11-10 16:38:24,184] Trial 0 finished with value: 0.9272511324215146 and parameters: {'w_0': 0.17490435839951257, 'w_1': 0.5405763345114564, 'w_2': 0.8556252950688278, 'w_3': 0.11200279850431538, 'w_4': 0.8863898238199533, 'w_5': 0.4744376211211664, 'w_6': 0.4606417205650014, 'w_7': 0.5785218958247634, 'w_8': 0.8109314882566101, 'w_9': 0.8721583540766153, 'w_10': 0.14101930201167534, 'w_11': 0.2900963588972443, 'w_12': 0.09096234606875975, 'w_13': 0.27353005231502425, 'w_14': 0.1199767684766394, 'w_15': 0.3105786751087487, 'w_16': 0.1254637002691361, 'w_17': 0.7362507814026968, 'w_18': 0.33931744047148416, 'w_19': 0.5404095344810235, 'w_20': 0.34045997865802813, 'w_21': 0.9739115821682366, 'w_22': 0.42957469047939356, 'w_23': 0.054913391373575426, 'w_24': 0.6931151799533075, 'w_25': 0.9269616839760974, 'w_26': 0.9940177974240783, 'w_27': 0.3065458373176787, 'w_28': 0

In [15]:
# 1. Get the best trial from your study
best_trial = study.best_trial

# 2. Extract the weights corresponding to the best trial's parameters
best_params = best_trial.params
raw_best_weights = np.array([best_params[f"w_{i}"] for i in range(num_models)])


In [16]:
# 3. Normalize the weights so they sum to 1 (matching your objective function logic)
sum_w = np.sum(raw_best_weights)
if sum_w == 0:
  final_weights = raw_best_weights
else:
  final_weights = raw_best_weights / sum_w


In [17]:
# 4. Print the best validation score and weights
print(f"Best Optuna ROC-AUC Score: {best_trial.value:.4f}")
print(f"Optimized Weights: {np.round(final_weights, 4)}")


Best Optuna ROC-AUC Score: 0.9469
Optimized Weights: [0.0255 0.0044 0.0016 0.0108 0.0106 0.0098 0.056  0.0077 0.023  0.0636
 0.0159 0.0259 0.0101 0.004  0.0052 0.0595 0.0099 0.0016 0.0045 0.0433
 0.0001 0.0035 0.0364 0.0619 0.0259 0.0698 0.0668 0.0649 0.058  0.0475
 0.0001 0.0516 0.0502 0.0055 0.034  0.0047 0.0259]


In [18]:
# 5. Compute the final blended predictions on the test set
# X_test contains your test OOF columns (excluding 'id')
final_blend_test = np.dot(X_test.values, final_weights)

# 6. Assign to your submission dataframe and save
submit["is_fraud"] = final_blend_test
submit.to_csv(competition_path / "Blending Optuna v4.csv", index=False)
print("Submission file successfully saved!")

Submission file successfully saved!
